In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import random_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
dataset_path = "dataset/pomegranate_diseases_dataset"
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])


dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)

In [3]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size


train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [5]:
model = nn.Sequential(
    nn.Conv2d(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(
        in_channels=16,
        out_channels=32,
        kernel_size=3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(32*32*32,100),
    nn.ReLU(),

    nn.Linear(100,32),
    nn.ReLU(),

    nn.Linear(32,5)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

In [6]:
def train(x,y):
    output = model(x)
    loss = criterion(output,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [7]:
epoch = 5
for k in range(epoch):
    batches = 0
    imgs = 0
    total_loss = 0
    for images,labels in train_loader:
        x = images
        y = labels
        total_loss = total_loss + train(x,y)
        batches = batches + 1
        imgs = imgs + 32
        print("\rImages:",imgs,end="")
    
    print(f"\nEpoch : {k+1} || Loss : {total_loss/(batches)}")

Images: 4096
Epoch : 1 || Loss : 0.9672881341539323
Images: 4096
Epoch : 2 || Loss : 0.4888106206199154
Images: 4096
Epoch : 3 || Loss : 0.32192434393800795
Images: 4096
Epoch : 4 || Loss : 0.20383430796209723
Images: 4096
Epoch : 5 || Loss : 0.15982181970321108


In [12]:
image, label = test_dataset[112]

model.eval()

with torch.no_grad():
    output = model(image.unsqueeze(0))
    prediction = torch.argmax(output, dim=1)

print("Actual:", test_dataset.dataset.classes[label])
print("Predicted:", test_dataset.dataset.classes[prediction.item()])

Actual: Cercospora
Predicted: Cercospora


In [14]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)

        predictions = torch.argmax(output, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100

print(f"Test Accuracy: {accuracy:.2f} %")

Test Accuracy: 92.65 %


In [ ]:
model.eval()
correct = 0

Alternaria = 0
Anthracnose = 0
Bacterial_Blight = 0
Cercospora = 0
Healthy = 0

Alternaria_corr = 0
Anthracnose_corr = 0
Bacterial_Blight_corr = 0
Cercospora_corr = 0
Healthy_corr = 0

total = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)

        predictions = torch.argmax(output, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100

print(f"Test Accuracy: {accuracy:.2f} %")

In [15]:
torch.save(model.state_dict(), "model.pth")